/kaggle/input/playground-series-s4e4/sample_submission.csv
/kaggle/input/playground-series-s4e4/train.csv
/kaggle/input/playground-series-s4e4/test.csv


#  LOADING DATA AND IT'S ANALYSIS 

In [2]:
# loading train and test data
train_data=pd.read_csv('/kaggle/input/playground-series-s4e4/train.csv')
test_data=pd.read_csv('/kaggle/input/playground-series-s4e4/test.csv')

In [3]:
# viewing train data
train_data

,id,Sex,Length,Diameter,Height,Whole weight,Whole weight.1,Whole weight.2,Shell weight,Rings
0,0,F,0.550,0.430,0.150,0.7715,0.3285,0.1465,0.2400,11
1,1,F,0.630,0.490,0.145,1.1300,0.4580,0.2765,0.3200,11
2,2,I,0.160,0.110,0.025,0.0210,0.0055,0.0030,0.0050,6
3,3,M,0.595,0.475,0.150,0.9145,0.3755,0.2055,0.2500,10
4,4,I,0.555,0.425,0.130,0.7820,0.3695,0.1600,0.1975,9
...,...,...,...,...,...,...,...,...,...,...
90610,90610,M,0.335,0.235,0.075,0.1585,0.0685,0.0370,0.0450,6
90611,90611,M,0.555,0.425,0.150,0.8790,0.3865,0.1815,0.2400,9
90612,90612,I,0.435,0.330,0.095,0.3215,0.1510,0.0785,0.0815,6
90613,90613,I,0.345,0.270,0.075,0.2000,0.0980,0.0490,0.0700,6


In [4]:
# looking for any redundant entries in train data
train_data.duplicated().sum()

0

In [5]:
# Check for missing values in target column
train_data.Rings.isnull().sum()

0

In [6]:
# Check for missing values and datatypes of features
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90615 entries, 0 to 90614
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              90615 non-null  int64  
 1   Sex             90615 non-null  object 
 2   Length          90615 non-null  float64
 3   Diameter        90615 non-null  float64
 4   Height          90615 non-null  float64
 5   Whole weight    90615 non-null  float64
 6   Whole weight.1  90615 non-null  float64
 7   Whole weight.2  90615 non-null  float64
 8   Shell weight    90615 non-null  float64
 9   Rings           90615 non-null  int64  
dtypes: float64(7), int64(2), object(1)
memory usage: 6.9+ MB


In [7]:
# Check for missing values and datatypes in test data
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60411 entries, 0 to 60410
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   id              60411 non-null  int64  
 1   Sex             60411 non-null  object 
 2   Length          60411 non-null  float64
 3   Diameter        60411 non-null  float64
 4   Height          60411 non-null  float64
 5   Whole weight    60411 non-null  float64
 6   Whole weight.1  60411 non-null  float64
 7   Whole weight.2  60411 non-null  float64
 8   Shell weight    60411 non-null  float64
dtypes: float64(7), int64(1), object(1)
memory usage: 4.1+ MB


In [8]:
# distribution of feature and target values
train_data.describe().T

,count,mean,std,min,25%,50%,75%,max
id,90615.0,45307.000000,26158.441658,0.0000,22653.5000,45307.0000,67960.5000,90614.0000
Length,90615.0,0.517098,0.118217,0.0750,0.4450,0.5450,0.6000,0.8150
Diameter,90615.0,0.401679,0.098026,0.0550,0.3450,0.4250,0.4700,0.6500
Height,90615.0,0.135464,0.038008,0.0000,0.1100,0.1400,0.1600,1.1300
Whole weight,90615.0,0.789035,0.457671,0.0020,0.4190,0.7995,1.0675,2.8255
Whole weight.1,90615.0,0.340778,0.204428,0.0010,0.1775,0.3300,0.4630,1.4880
Whole weight.2,90615.0,0.169422,0.100909,0.0005,0.0865,0.1660,0.2325,0.7600
Shell weight,90615.0,0.225898,0.130203,0.0015,0.1200,0.2250,0.3050,1.0050
Rings,90615.0,9.696794,3.176221,1.0000,8.0000,9.0000,11.0000,29.0000


In [9]:
# creating copy of datasets to avoid alteration(s) in original data
tr_data=train_data.copy()
tt_data=test_data.copy()

In [16]:
# splitting data into test and valid
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [17]:
# segerating numerical columns
numerical_cols = X.select_dtypes(include=['int', 'float']).columns.tolist()
numerical_cols

['id',
 'Length',
 'Diameter',
 'Height',
 'Whole weight',
 'Whole weight.1',
 'Whole weight.2',
 'Shell weight',
 'Sex_encoded']

In [18]:
# categorical_cols = tr_data.select_dtypes(include='object').columns.tolist()
# categorical_cols

In [20]:
X_train_ready=pd.DataFrame(preprocessor.fit_transform(X_train))
X_valid_ready=pd.DataFrame(preprocessor.transform(X_valid))

tt_ready=pd.DataFrame(preprocessor.transform(tt_data))
 
print(X_train_ready.shape)
print(X_valid_ready.shape)
print(tt_ready.shape)

(72492, 9)
(18123, 9)
(60411, 9)


In [21]:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,mean_squared_log_error
# from sklearn.metrics import root_mean_squared_log_error
import math
import optuna

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

In [22]:
# Define objective function
def objective(trial):
    # Suggest values for hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-3, 0.1, log=True)
    depth = trial.suggest_int("depth", 1, 10)
    subsample = trial.suggest_float("subsample", 0.05, 1.0)
    colsample_bylevel = trial.suggest_float("colsample_bylevel", 0.05, 1.0)
    min_data_in_leaf = trial.suggest_int("min_data_in_leaf", 1, 100)
   
    # Create and fit catboost model
    model = CatBoostRegressor(
    learning_rate = learning_rate,
    depth = depth,
    subsample = subsample,
    colsample_bylevel = colsample_bylevel,
        min_data_in_leaf = min_data_in_leaf,
    random_state=42,
    silent=True
    )
    model.fit(X_train, y_train)

    # Make predictions and calculate RMSE
    y_pred = model.predict(X_valid)
    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    mae = mean_absolute_error(y_valid, y_pred)
    r2 = r2_score(y_valid, y_pred)
    msle = mean_squared_log_error(y_valid,y_pred)
    rmsle = math.sqrt(msle)

    score = rmsle
    return score

In [23]:
# Create study object
study = optuna.create_study(direction="minimize")

# Run optimization process
study.optimize(objective, n_trials=20, show_progress_bar=True)

[I 2024-04-27 16:16:05,079] A new study created in memory with name: no-name-e7693df2-1d07-4a0b-a671-c0afaabc2615


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2024-04-27 16:16:15,237] Trial 0 finished with value: 0.16300028077454054 and parameters: {'learning_rate': 0.002529053511005426, 'depth': 7, 'subsample': 0.6866302950002945, 'colsample_bylevel': 0.9751988917665163, 'min_data_in_leaf': 57}. Best is trial 0 with value: 0.16300028077454054.
[I 2024-04-27 16:16:20,548] Trial 1 finished with value: 0.1784581742093397 and parameters: {'learning_rate': 0.0022246455141379895, 'depth': 8, 'subsample': 0.19294735691079223, 'colsample_bylevel': 0.21651515767936075, 'min_data_in_leaf': 52}. Best is trial 0 with value: 0.16300028077454054.
[I 2024-04-27 16:16:28,395] Trial 2 finished with value: 0.1506893037965688 and parameters: {'learning_rate': 0.06520178699228539, 'depth': 6, 'subsample': 0.5116299469690008, 'colsample_bylevel': 0.560208318807572, 'min_data_in_leaf': 71}. Best is trial 2 with value: 0.1506893037965688.
[I 2024-04-27 16:16:36,924] Trial 3 finished with value: 0.15345077728728856 and parameters: {'learning_rate': 0.0163291709

In [24]:
# Print best trial and best hyperparameters
print("Best trial:", study.best_trial)
print("Best hyperparameters:", study.best_params)
best_params = study.best_params

Best trial: FrozenTrial(number=2, state=TrialState.COMPLETE, values=[0.1506893037965688], datetime_start=datetime.datetime(2024, 4, 27, 16, 16, 20, 552920), datetime_complete=datetime.datetime(2024, 4, 27, 16, 16, 28, 394900), params={'learning_rate': 0.06520178699228539, 'depth': 6, 'subsample': 0.5116299469690008, 'colsample_bylevel': 0.560208318807572, 'min_data_in_leaf': 71}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.1, log=True, low=0.001, step=None), 'depth': IntDistribution(high=10, log=False, low=1, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.05, step=None), 'colsample_bylevel': FloatDistribution(high=1.0, log=False, low=0.05, step=None), 'min_data_in_leaf': IntDistribution(high=100, log=False, low=1, step=1)}, trial_id=2, value=None)
Best hyperparameters: {'learning_rate': 0.06520178699228539, 'depth': 6, 'subsample': 0.5116299469690008, 'colsample_bylevel': 0.560208318807572, 'min_da

In [25]:
best_learning_rate= best_params['learning_rate']
best_depth= best_params['depth']
best_subsample= best_params['subsample']
best_colsample_bylevel= best_params['colsample_bylevel']
best_min_data_in_leaf= best_params['min_data_in_leaf']


best_model = CatBoostRegressor(
    learning_rate = best_learning_rate,
    depth = best_depth,
    subsample = best_subsample,
    colsample_bylevel = best_colsample_bylevel,
    min_data_in_leaf = best_min_data_in_leaf,
    random_state=42,
    silent=True
    )

In [26]:
best_model.fit(X_train,y_train)

In [27]:
predictions = best_model.predict(tt_data)
# Save test predictions to file
output = pd.DataFrame({'id': tt_data.id,
                       'Rings': predictions})
output.to_csv('submission.csv', index=False)

In [28]:
output

,id,Rings
0,90615,9.340154
1,90616,9.540422
2,90617,10.329467
3,90618,10.274242
4,90619,7.592693
...,...,...
60406,151021,6.406652
60407,151022,9.696943
60408,151023,11.909659
60409,151024,14.131646
